# 🚀 IHDP Agent for F-16 Aircraft Control

This example demonstrates the use of an **Incremental Heuristic Dynamic Programming (IHDP)** agent for controlling the longitudinal motion of an F-16 aircraft.

## 📋 What we will do:
1. Set up the F-16 simulation environment
2. Create and configure the IHDP agent
3. Run the control simulation
4. Visualize the results

---

**Paper-equation update:** the SISO actor gradient now includes the physical output scale. The actor learning rate and its floor are expressed in these units. Previous cached results were cleared; execute this notebook to obtain results with the current implementation.

## 📦 Library Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from tqdm import tqdm

# TensorAeroSpace imports
from tensoraerospace.envs.f16.linear_longitudinal import LinearLongitudinalF16
from tensoraerospace.utils import generate_time_period, convert_tp_to_sec_tp
from tensoraerospace.signals.standard import unit_step
from tensoraerospace.agent.ihdp.model import IHDPAgent
from tensoraerospace.benchmark import ControlBenchmark

# Configure matplotlib for nicer plots
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print('✅ All libraries successfully imported!')

## ⚙️ Simulation Parameters

Configure the main parameters for our simulation:

In [ ]:
# 🕐 Time parameters
dt = 0.01  # Discretization step (seconds)
simulation_time = 20  # Total simulation time (seconds)
tp = generate_time_period(tn=simulation_time, dt=dt)
tps = convert_tp_to_sec_tp(tp, dt=dt)
number_time_steps = len(tp)

# 📊 Reference signal (step signal for angle of attack)
reference_signals = np.reshape(
    unit_step(degree=5, tp=tp, time_step=1000*dt, output_rad=True), 
    [1, -1]
)

print(f'📈 Simulation parameters:')
print(f'   • Simulation time: {simulation_time} sec')
print(f'   • Discretization step: {dt} sec')
print(f'   • Number of steps: {number_time_steps}')
print(f'   • Reference signal shape: {reference_signals.shape}')

## 🛩️ Creating the F-16 Environment

Set up the simulation environment for F-16 longitudinal motion:

In [ ]:
# 🎯 Initial conditions
initial_state = [[0], [0], [0]]  # [theta, alpha, q]

# 🏗️ Create the environment
env = gym.make(
    'LinearLongitudinalF16-v0',
    number_time_steps=number_time_steps,
    initial_state=initial_state,
    reference_signal=reference_signals,
    use_reward=False,
    state_space=["theta", "alpha", "q"],
    output_space=["theta", "alpha", "q"],
    control_space=["ele"],
    tracking_states=["alpha"]
)

# Reset the environment
initial_observation, info = env.reset()

print(f'🛩️ F-16 environment created successfully!')
print(f'   • State space: {env.unwrapped.state_space}')
print(f'   • Tracked states: {env.unwrapped.tracking_states}')
print(f'   • Control inputs: {env.unwrapped.control_space}')
print(f'   • Initial state: {initial_observation.flatten()}')

## 🤖 IHDP Agent Configuration

Configure the parameters for the Actor, Critic, and Incremental Model:

In [ ]:
# 🎭 Actor settings (control policy)
actor_settings = {
    "start_training": 5,                    # Training start
    "layers": (25, 1),                     # Neural network architecture
    "activations": ('tanh', 'tanh'),       # Activation functions
    "learning_rate": 2.0 / 25.0,
    "learning_rate_min": 0.001 / 25.0,                   # Learning rate
    "learning_rate_exponent_limit": 10,   # Exponent limit
    "type_PE": "combined",                # Excitation signal type
    "amplitude_3211": 15,                 # 3-2-1-1 signal amplitude
    "pulse_length_3211": 5/dt,            # Pulse length
    "maximum_input": 25,                  # Maximum input
    "maximum_q_rate": 20,                 # Maximum q rate
    "WB_limits": 30,                     # Weight limits
    "NN_initial": 120,                   # Initial initialization
    "cascade_actor": False,               # Cascade actor
    "learning_rate_cascaded": 1.2         # Cascade learning rate
}

print('🎭 Actor settings ready!')

In [ ]:
# 🎯 Critic settings (value estimation)
critic_settings = {
    "Q_weights": [8],                     # Quality function weights
    "start_training": -1,                 # Critic training start
    "gamma": 0.99,                        # Discount factor
    "learning_rate": 15,                  # Learning rate
    "learning_rate_exponent_limit": 10,   # Exponent limit
    "layers": (25, 1),                    # Neural network architecture
    "activations": ("tanh", "linear"),    # Activation functions
    "WB_limits": 30,                     # Weight limits
    "NN_initial": 120,                   # Initial initialization
    "indices_tracking_states": env.unwrapped.indices_tracking_states
}

print('🎯 Critic settings ready!')

In [ ]:
# ⚡ Incremental model settings
incremental_settings = {
    "number_time_steps": number_time_steps,  # Number of steps
    "dt": dt,                              # Discretization step
    "input_magnitude_limits": 25,          # Amplitude limits
    "input_rate_limits": 60,               # Rate limits
}

print('⚡ Incremental model settings ready!')

In [ ]:
# 🤖 Create the IHDP agent
print('🔧 Creating IHDP agent...')

model = IHDPAgent(
    actor_settings,
    critic_settings,
    incremental_settings,
    env.unwrapped.tracking_states,
    env.unwrapped.state_space,
    env.unwrapped.control_space,
    number_time_steps,
    env.unwrapped.indices_tracking_states
)

print('✅ IHDP agent successfully created!')

## 🚀 Running the Simulation

Now let's run the control simulation and collect data:

In [ ]:
# 📊 Initialize arrays for data collection
states_history = []
controls_history = []
rewards_history = []
time_history = []

# 🎯 Initial state
xt = np.array([[0], [0], [0]])

print('🚀 Starting simulation...')
print(f'📊 {number_time_steps-3} steps will be executed')

# 🔄 Main simulation loop
for step in tqdm(range(number_time_steps-3), desc="🎮 Simulation", ncols=100):
    # Get the control signal from the agent
    ut = model.predict(xt, reference_signals, step)
    
    # Step the environment
    xt, reward, terminated, truncated, info = env.step(np.array(ut))
    
    # Save data
    states_history.append(xt.copy())
    controls_history.append(ut.copy())
    rewards_history.append(reward)
    time_history.append(step * dt)
    
    # Check for termination
    if terminated or truncated:
        break

print('✅ Simulation finished successfully!')
print(f'📈 Collected {len(states_history)} data points')

## 📊 Results Visualization

Build detailed plots to analyze the results:

In [ ]:
# 🔄 Process data for visualization
if len(states_history) > 0:
    # Convert to numpy arrays
    states_array = np.array([s.flatten() for s in states_history])
    controls_array = np.array([c.flatten() for c in controls_history])
    time_array = np.array(time_history)
    
    # Extract individual states
    theta = states_array[:, 0]  # Pitch angle
    alpha = states_array[:, 1]  # Angle of attack
    q = states_array[:, 2]      # Pitch rate
    
    # Control signal
    elevator = controls_array[:, 0] if len(controls_array) > 0 else []
    
    # Reference signal for comparison
    reference_alpha = reference_signals[0, :len(alpha)]
    
    print('📊 Data processed and ready for visualization')
else:
    print('❌ No data available for visualization')

In [ ]:
# 🎨 Build the main plots
if len(states_history) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('🛩️ F-16 control results with the IHDP agent', fontsize=16, fontweight='bold')
    
    # Plot 1: Angle-of-attack tracking
    axes[0, 0].plot(time_array, alpha, 'b-', linewidth=2, label='Actual α')
    axes[0, 0].plot(time_array, reference_alpha, 'r--', linewidth=2, label='Reference α')
    axes[0, 0].set_xlabel('Time (s)')
    axes[0, 0].set_ylabel('Angle of attack α (rad)')
    axes[0, 0].set_title('📈 Angle-of-attack tracking')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Plot 2: Pitch angle
    axes[0, 1].plot(time_array, theta, 'g-', linewidth=2, label='Pitch angle θ')
    axes[0, 1].set_xlabel('Time (s)')
    axes[0, 1].set_ylabel('Pitch angle θ (rad)')
    axes[0, 1].set_title('📐 Pitch angle')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Plot 3: Pitch rate
    axes[1, 0].plot(time_array, q, 'm-', linewidth=2, label='Pitch rate q')
    axes[1, 0].set_xlabel('Time (s)')
    axes[1, 0].set_ylabel('Pitch rate q (rad/s)')
    axes[1, 0].set_title('🔄 Pitch rate')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Plot 4: Control signal
    if len(elevator) > 0:
        axes[1, 1].plot(time_array, elevator, 'orange', linewidth=2, label='Elevator')
        axes[1, 1].set_xlabel('Time (s)')
        axes[1, 1].set_ylabel('Elevator deflection (rad)')
        axes[1, 1].set_title('🎛️ Control signal')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print('🎨 Main plots built!')

In [ ]:
# 📊 Tracking error analysis
if len(states_history) > 0:
    # Compute the error
    tracking_error = alpha - reference_alpha
    
    # Error plot
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.plot(time_array, tracking_error, 'r-', linewidth=2)
    plt.axhline(y=0, color='k', linestyle='--', alpha=0.5)
    plt.xlabel('Time (s)')
    plt.ylabel('Tracking error (rad)')
    plt.title('📉 Angle-of-attack tracking error')
    plt.grid(True, alpha=0.3)
    
    # Error histogram
    plt.subplot(1, 2, 2)
    plt.hist(tracking_error, bins=30, alpha=0.7, color='skyblue', edgecolor='black')
    plt.xlabel('Tracking error (rad)')
    plt.ylabel('Frequency')
    plt.title('📊 Error distribution')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print('📊 Error analysis finished!')

## 📈 Performance Statistics

Analyze the control quality:

In [ ]:
# 📊 Compute statistics
if len(states_history) > 0:
    # Error metrics
    mae = np.mean(np.abs(tracking_error))  # Mean absolute error
    rmse = np.sqrt(np.mean(tracking_error**2))  # Root mean squared error
    max_error = np.max(np.abs(tracking_error))  # Maximum error
    std_error = np.std(tracking_error)  # Standard deviation
    
    # Control metrics
    if len(elevator) > 0:
        max_control = np.max(np.abs(elevator))
        mean_control = np.mean(np.abs(elevator))
        control_variation = np.std(elevator)
    
    # Settling time (time to reach 95% of the target value)
    target_value = reference_alpha[-1]
    settling_threshold = 0.05 * abs(target_value)
    settling_indices = np.where(np.abs(alpha - target_value) <= settling_threshold)[0]
    settling_time = time_array[settling_indices[0]] if len(settling_indices) > 0 else None
    
    # Pretty-print statistics
    print('' + '='*60)
    print('📊 CONTROL SYSTEM PERFORMANCE STATISTICS')
    print('='*60)
    
    print('🎯 TRACKING ACCURACY METRICS:')
    print(f'   • Mean absolute error (MAE): {mae:.6f} rad ({np.degrees(mae):.3f}°)')
    print(f'   • Root mean squared error (RMSE): {rmse:.6f} rad ({np.degrees(rmse):.3f}°)')
    print(f'   • Maximum error: {max_error:.6f} rad ({np.degrees(max_error):.3f}°)')
    print(f'   • Standard deviation: {std_error:.6f} rad ({np.degrees(std_error):.3f}°)')
    
    if settling_time is not None:
        print(f'⏱️ DYNAMIC CHARACTERISTICS:')
        print(f'   • Settling time (95%): {settling_time:.2f} sec')
    
    if len(elevator) > 0:
        print('🎛️ CONTROL CHARACTERISTICS:')
        print(f'   • Maximum elevator deflection: {max_control:.6f} rad ({np.degrees(max_control):.3f}°)')
        print(f'   • Mean elevator deflection: {mean_control:.6f} rad ({np.degrees(mean_control):.3f}°)')
        print(f'   • Control variation: {control_variation:.6f} rad')
    
    print('⚡ OVERALL ASSESSMENT:')
    if mae < 0.01:
        print('   🟢 EXCELLENT: Very high tracking accuracy!')
    elif mae < 0.05:
        print('   🟡 GOOD: Acceptable tracking accuracy')
    else:
        print('   🔴 NEEDS IMPROVEMENT: Low tracking accuracy')
    
    print('='*60)

## 🎉 Conclusion

In this example we successfully:

- **Set up the simulation environment** for F-16 longitudinal motion
- **Created and trained an IHDP agent** with tuned parameters
- **Ran the control simulation** with reference signal tracking
- **Analyzed the results** with detailed performance statistics

### 🔧 Possible Improvements:

- Tune learning parameters for better convergence
- Add constraints on control signals
- Test with different reference signals
- Compare with other control methods

### 📚 Additional Resources:

- [TensorAeroSpace Documentation](https://tensoraerospace.readthedocs.io/)
- [Usage Examples](../README.md)
- [IHDP Theory](https://tensoraerospace.readthedocs.io/en/latest/agent/ihdp.html)

---

**🚀 Happy experimenting with TensorAeroSpace!**